# 第 11 章 サポートベクターマシンとカーネル法

できるだけ広い余白（マージン）を空ける境界線を選びます。カーネルで XOR も解きます。

対応する記事: [第 11 章 サポートベクターマシンとカーネル法（Polyglot Notebook（F#） の言語版）](../../../docs/article/grokking-machine-learning/fsharp/ch11.md)

実装本体: `apps/grokking-ml-fsharp/src/`

## セットアップ

実装本体（`../src/GrokkingMl/`）を `#load` で読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

VS Code の [Polyglot Notebooks 拡張](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) で開くか、Jupyter に .NET Interactive カーネルを登録して実行します。

```bash
dotnet tool install -g Microsoft.dotnet-interactive
dotnet interactive jupyter install
jupyter lab notebooks/
```

In [1]:
#load "../src/GrokkingMl/Ch05Perceptron.fs"
#load "../src/GrokkingMl/Ch11Svm.fs"

open GrokkingMl.Ch11Svm

## パーセプトロンのマージンは 0

第 5 章のパーセプトロンは、分離できた時点で更新を止めます。**その境界線がぎりぎりでも構いません。**

実際に測ると、境界線の上にちょうど乗っている点があります。正解率は 1.0 なのに、その点が少しでも動けば誤分類になります。

In [2]:
let points: Point list =
    [ [ 1.0; 0.0 ]; [ 0.0; 2.0 ]; [ 1.0; 1.0 ]; [ 1.0; 2.0 ]
      [ 1.0; 3.0 ]; [ 2.0; 2.0 ]; [ 2.0; 3.0 ]; [ 3.0; 2.0 ] ]

let labels = [ -1; -1; -1; -1; 1; 1; 1; 1 ]
let perceptronLabels = [ 0; 0; 0; 0; 1; 1; 1; 1 ]

let perceptron, _ =
    GrokkingMl.Ch05Perceptron.perceptronAlgorithm 0.01 1000 0 points perceptronLabels

let asSvm = { Weights = perceptron.Weights; Bias = perceptron.Bias }

printfn "パーセプトロン 正解率 %.2f  マージン %.4f" (accuracy asSvm points labels) (margin asSvm points)

パーセプトロン 正解率 

1.00

  マージン 

0.0000

## SVM は余白を稼ぐ

**正解率は同じ 1.0 でも、マージンがまったく違います。** ヒンジ損失が「正解しているのにマージンの内側にいる点」も押し返すためです。

In [3]:
let svm, errors = trainSvmWith 0.01 20000 0.01 0 points labels

printfn "重み   %A" (svm.Weights |> List.map (sprintf "%.4f"))
printfn "バイアス %.4f" svm.Bias
printfn "正解率  %.2f" (accuracy svm points labels)
printfn "マージン %.4f" (margin svm points)

重み   

["1.6888"; "1.6952"]

バイアス 

-5.7700

正解率  

1.00

マージン 

0.5774

## ヒンジ損失は「正解しているのに損失が残る」

第 5 章のパーセプトロン誤差は正解した点を無視しました。**ヒンジ損失はマージンの外に出るまで押し続けます。**

In [4]:
let sample = { Weights = [ 1.0; 1.0 ]; Bias = -3.0 }

printfn "%-12s %8s %6s %8s" "点" "スコア" "ラベル" "損失"

for point in [ [ 3.0; 2.0 ]; [ 1.5; 2.0 ]; [ 2.0; 1.0 ]; [ 1.0; 1.0 ] ] do
    printfn "%-12s %8.1f %6d %8.2f" (sprintf "%A" point) (score sample point) 1 (hingeLoss sample point 1)

点           

     スコア

   ラベル

      損失

[3.0; 2.0]  

...

    1.00

[1.0; 1.0]  

    -1.0

     1

    2.00

## カーネルで XOR を解く

**アルゴリズムは一切変えず、カーネル関数を差し替えるだけ** で XOR が解けます。線形カーネル（ただの内積）では解けません。

In [5]:
let xorPoints: Point list = [ [ 0.0; 0.0 ]; [ 0.0; 1.0 ]; [ 1.0; 0.0 ]; [ 1.0; 1.0 ] ]
let xorLabels = [ -1; 1; 1; -1 ]

for (name, kernel) in [ "線形", linearKernel
                        "多項式 2 次", polynomialKernel 2 1.0
                        "RBF", rbfKernel 1.0 ] do
    let m = trainKernelClassifier kernel xorPoints xorLabels
    printfn "%-12s 正解率 %.2f" name (kernelAccuracy m xorPoints xorLabels)

線形          

 正解率 

0.50

多項式 2 次     

 正解率 

1.00

RBF         

 正解率 

1.00

## 試してみる: 正則化とマージン

**直感に反しますが、正則化を強くするとマージンは狭くなります。** 重みを潰しすぎると、境界線からの距離そのものが縮むためです。

In [6]:
for strength in [ 0.005; 0.01; 0.05; 0.1; 0.5 ] do
    let m, _ = trainSvmWith 0.01 20000 strength 0 points labels
    printfn "λ = %-6f マージン %.4f  正解率 %.2f" strength (margin m points) (accuracy m points labels)

λ = 

0.005000

 マージン 

0.6116

  正解率 

...

0.500000

 マージン 

0.5248

  正解率 

1.00